<a href="https://colab.research.google.com/github/command404/My-First-Pipeline/blob/main/scikit_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

train_df = pd.read_csv('/content/drive/MyDrive/housing.csv')
test_df = pd.read_csv('/content/drive/MyDrive/housing.csv')


train_df = train_df.drop("ocean_proximity", axis = 1)
test_df = test_df.drop("ocean_proximity", axis = 1)
train_df = train_df.dropna()
test_df = test_df.dropna()

train_df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0


In [ ]:
test_df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0


In [ ]:
x_train,y_train = train_df.to_numpy()[: ,:-1], train_df.to_numpy()[: ,-1]
x_test,y_test = test_df.to_numpy()[: ,:-1], test_df.to_numpy()[: ,-1]
x_train.shape,y_train.shape,x_test.shape,y_test.shape

#to_numoy converts the dataset to a numoy (numpy.ndarray)
#: this means include all the row and column :-1 means include all the rows except the last column

((20433, 8), (20433,), (20433, 8), (20433,))

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, FunctionTransformer
from copy import deepcopy
#minmaxscaler converts values to either 0 or 1 ,gemini standardscaler
std_scaler = StandardScaler().fit(x_train[: ,:2])
min_max_scaler = MinMaxScaler().fit(x_train[: ,2:])
#in standard scaler we excluded the first 2 columns because std scaler is not that great with negative values
#minmaxscaler is better with negative values
def preprocessor(X):
  A = np.copy(X)
  A[: , :2] = std_scaler.transform(X[: , :2])
  A[: , 2:] = min_max_scaler.transform(X[: , 2:])
  return A
#definition of standard scaler:
#StandardScaler is a preprocessing tool that "levels the playing field" for your data by transforming features
#so they share a common scale. It centers each column’s average (mean) at 0 and scales the spread (standard deviation)
#to 1, effectively converting raw values into Z-scores. This is crucial because many machine learning algorithms—like
#SVMs or Neural Networks—calculate distances between data points; without scaling, a feature with large numbers (like Income) would mathematically "overpower"
# a feature with small numbers (like Age), even if the smaller one is more important. By squashing all features into this standard normal distribution,
#you ensure the model treats every variable with equal architectural importance and reaches an optimal solution much faster.



#definition of MinMaxScaler:
#MinMaxScaler is a normalization tool that "squashes" all your data points into a specific,
#fixed range—usually between 0 and 1. Unlike StandardScaler, which centers data around an average,
#MinMaxScaler identifies the minimum and maximum values of a column and stretches or compresses everything
#in between so that the smallest value becomes exactly 0 and the largest becomes exactly 1.

In [ ]:
preprocessor(x_test)
x_test

array([[-1.2223e+02,  3.7880e+01,  4.1000e+01, ...,  3.2200e+02,
         1.2600e+02,  8.3252e+00],
       [-1.2222e+02,  3.7860e+01,  2.1000e+01, ...,  2.4010e+03,
         1.1380e+03,  8.3014e+00],
       [-1.2224e+02,  3.7850e+01,  5.2000e+01, ...,  4.9600e+02,
         1.7700e+02,  7.2574e+00],
       ...,
       [-1.2122e+02,  3.9430e+01,  1.7000e+01, ...,  1.0070e+03,
         4.3300e+02,  1.7000e+00],
       [-1.2132e+02,  3.9430e+01,  1.8000e+01, ...,  7.4100e+02,
         3.4900e+02,  1.8672e+00],
       [-1.2124e+02,  3.9370e+01,  1.6000e+01, ...,  1.3870e+03,
         5.3000e+02,  2.3886e+00]])

In [ ]:
preprocess_transformer = FunctionTransformer(preprocessor)
#were using this to make specific transformations in the dataset
preprocess_transformer
# definition of function transformer:
#A FunctionTransformer is a "DIY" tool that allows you to turn any custom Python function into
#a formal Scikit-Learn transformation step. While tools like StandardScaler have built-in math,
#a FunctionTransformer lets you apply your own logic—such as taking the logarithm of a skewed column,
#calculating the ratio between two features (like "rooms per household"), or even a simple cleanup
#task—and wrap it in a way that fits perfectly inside a Pipeline. This is incredibly powerful because it
#ensures your custom data "tweaks" are applied automatically and consistently during both training and
#testing, preventing you from having to manually clean new data every time you want a prediction.

FunctionTransformer(func=<function preprocessor at 0x7bf8ebc76520>)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
#im not sure if a pipeline can fit a model so we first have to fit the pipeline and then we transform and train the
#the model using the pipeline
p1 = Pipeline([('Scaler', preprocess_transformer),
               ('linear regression', LinearRegression())])
#scaler is the name of the pipeline(we gave the name)
#preprocess_transform transform the data (basically drop columns and stuff we defined what columns to drop in the start and in the middle)
#linear rgression is basically the type of model its gonna train
p1

Pipeline(steps=[('Scaler',
                 FunctionTransformer(func=<function preprocessor at 0x7bf8ebc76520>)),
                ('linear regression', LinearRegression())])

In [ ]:
from sklearn.metrics import mean_absolute_error
#mean_absolute_error is used to look at the errors in our model
def fit_and_print(p, x_train=x_train, x_test=x_test, y_test=y_test, y_train=y_train):
  p.fit(x_train, y_train)#we will first fit the pipeline
  train_preds = p.predict(x_train)
  test_preds = p.predict(x_test)
  print('training error: ' + str(mean_absolute_error(train_preds, y_train)))
  print('testing error: ' + str(mean_absolute_error(test_preds, y_test)))
  #this basically fits the model scalers it using the pipeline and then train a linear regression model and then give us its errors

In [ ]:
fit_and_print(p1)
#the 50799 is the dollars we were off by in the dataset

training error: 50799.63072895291
testing error: 50799.63072895291


In [ ]:
from sklearn.neighbors import KNeighborsRegressor as KNR

p2 = Pipeline([('Scaler', preprocess_transformer),
               ('KNR regression', KNR())])
fit_and_print(p2)

training error: 28095.267508442226
testing error: 28095.267508442226


In [ ]:
from sklearn.linear_model import LogisticRegression
p3 = Pipeline([('Scaler', preprocess_transformer),
               ('Logistic Regression', LogisticRegression())])
fit_and_print(p3)

training error: 149216.51544070867
testing error: 149216.51544070867


In [ ]:
from sklearn.ensemble import RandomForestRegressor
p4 = Pipeline([('Scaler', preprocess_transformer),
               ('Random Forest classifier', RandomForestRegressor())])
fit_and_print(p4)

training error: 11543.36124161895
testing error: 11543.36124161895


In [ ]:
from sklearn.neural_network import MLPRegressor
mlpr = MLPRegressor(hidden_layer_sizes = (11,11,11), max_iter = 500)
p5 = Pipeline([('scaler', preprocess_transformer),
               ('neural network', mlpr)])
fit_and_print(p5)

training error: 45557.386489961114
testing error: 45557.386489961114


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


In [ ]:
from sklearn.svm import SVC
p6 = Pipeline([('scaler', preprocess_transformer),
               ('SVM', SVC())])
fit_and_print(p6)

In [ ]:
from sklearn.tree import DecisionTreeRegressor
p7 = Pipeline([('scaler', preprocess_transformer),
               ('decision tree regressor', DecisionTreeRegressor())])
fit_and_print(p7)